In [16]:
%reload_ext autoreload
%autoreload 2

In [76]:
import os
import pickle
from pathlib import Path

import polars as pl
import numpy as np
import torch

from syncabel.embeddings import TextEncoder

torch.cuda.empty_cache()
device = "cuda" if torch.cuda.is_available() else "cpu"


# Load Encoder (can be SapBERT local or CODER from hub)
# Example local SapBERT: model_name = "SapBERT-from-PubMedBERT-fulltext"
# Example CODER from hub: model_name = "GanjinZero/coder-large-v3"
model_name = "coder_all"
dataset_name = "MEDLINE"

In [77]:
local_model_path = f"models/encoder/{model_name}"

encoder = TextEncoder(
    model_name=local_model_path if os.path.isdir(local_model_path) else model_name
)

# Create Embedding
def get_bert_embed(
    phrase_list,
    normalize=True,
    summary_method="CLS",
    tqdm_bar=True,
    batch_size=128,
):
    """Compatibility wrapper that forwards to TextEncoder.encode with CODER-like behavior."""
    embs = encoder.encode(
        list(phrase_list),
        batch_size=batch_size,
        normalize=normalize,
        summary_method=summary_method,
        tqdm_bar=tqdm_bar,
        max_length=32,
    )
    # Return a torch tensor on CPU to match previous behavior
    return torch.from_numpy(embs)



In [78]:
import torch

def hf_model_gpu_size_mb(model):
    return (
        sum(p.numel() * p.element_size() for p in model.parameters()) +
        sum(b.numel() * b.element_size() for b in model.buffers())
    ) / 1024**2
print(f"Model GPU size: {hf_model_gpu_size_mb(encoder.model):.2f} MB")

Model GPU size: 678.46 MB


In [79]:

# Load UMLS data
if dataset_name == "MedMentions":
    dataset_short = "MM"
elif dataset_name in ["EMEA", "MEDLINE"]:
    dataset_short = "QUAERO"
elif "SPACCC" in dataset_name:
    dataset_short = "SPACCC"
else:
    dataset_short = dataset_name

# UMLS Embeddings
umls_path = (
    Path("data") / "UMLS_processed" / dataset_short / "all_disambiguated.parquet"
)
umls_df = pl.read_parquet(umls_path)
code_col = "SNOMED_code" if "SNOMED_code" in umls_df.columns else "CUI"


In [80]:
import polars as pl
import numpy as np
from tqdm import tqdm
import torch
import gc

########################################
# CONFIG
########################################
BATCH_SIZE = 4096          # increase if GPU allows
EMB_DTYPE = np.float32   # or np.float16

########################################
# 1. PREPARE UMLS TABLE (NO EMBEDDINGS)
########################################

print("Preparing UMLS tables...")

df_all = (
    umls_df
    .group_by(["GROUP", "Entity"])
    .agg(pl.col(code_col).unique().alias("codes"))
    .sort(["GROUP", "Entity"])
)

########################################
# 2. BUILD GLOBAL UNIQUE SYNONYM LIST
########################################

all_synonyms = df_all["Entity"].unique().to_list()
print(f"Total unique synonyms: {len(all_synonyms)}")

########################################
# 3. COMPUTE EMBEDDINGS ONCE (GPU)
########################################

torch.cuda.empty_cache()
gc.collect()

all_embeddings = np.asarray(
    get_bert_embed(
        all_synonyms,
        batch_size=BATCH_SIZE
    ),
    dtype=EMB_DTYPE
)

assert len(all_embeddings) == len(all_synonyms)

########################################
# 4. MAP SYNONYM → EMBEDDING
########################################

syn2emb = dict(zip(all_synonyms, all_embeddings))

########################################
# 5. BUILD PER-GROUP UMLS TABLES (FAST)
########################################

umls_tables = {}

for group in tqdm(df_all["GROUP"].unique(), desc="Building group tables"):
    df_g = df_all.filter(pl.col("GROUP") == group)

    synonyms = df_g["Entity"].to_list()
    codes = df_g["codes"].to_list()

    embeddings = np.stack([syn2emb[s] for s in synonyms])

    umls_tables[group] = {
        "codes": codes,
        "emb": embeddings,
        "synonyms": synonyms
    }

print("UMLS tables built successfully.")

########################################
# DONE
########################################


Preparing UMLS tables...
Total unique synonyms: 6957811


Building group tables: 100%|##########| 10/10 [00:22<00:00,  2.29s/it]


UMLS tables built successfully.


In [81]:
import sys

def deep_getsizeof(obj, seen=None):
    if seen is None:
        seen = set()
    obj_id = id(obj)
    if obj_id in seen:
        return 0
    seen.add(obj_id)

    size = sys.getsizeof(obj)
    if hasattr(obj, "__dict__"):
        size += deep_getsizeof(obj.__dict__, seen)
    elif isinstance(obj, dict):
        size += sum(deep_getsizeof(k, seen) + deep_getsizeof(v, seen) for k, v in obj.items())
    elif isinstance(obj, (list, tuple, set)):
        size += sum(deep_getsizeof(i, seen) for i in obj)
    return size

In [48]:
trie_size = deep_getsizeof(umls_tables) / 1024**2
print(f"Trie candidate object size (deep): {trie_size:.2f} MB")

Trie candidate object size (deep): 1732.16 MB


In [23]:
trie_size = deep_getsizeof(umls_tables) / 1024**2
print(f"Trie candidate object size (deep): {trie_size:.2f} MB")

Trie candidate object size (deep): 21944.55 MB


In [8]:
trie_size = deep_getsizeof(umls_tables) / 1024**2
print(f"Trie candidate object size (deep): {trie_size:.2f} MB")

Trie candidate object size (deep): 15327.93 MB


In [10]:
def embedding_size_mb(embeddings: np.ndarray) -> float:
    """
    Exact memory size of a NumPy embedding matrix in MB.
    """
    return embeddings.nbytes / 1024**2

print(f"Embedding size: {embedding_size_mb(all_embeddings):.2f} MB")


Embedding size: 19011.64 MB


In [82]:
test_data = pl.read_csv(
    Path("data") / "final_data" /dataset_name / "test_tfidf_annotations.tsv",
    separator="\t",
    has_header=True,
    schema_overrides={
        "code": str,
        "mention_id": str,
        "filename": str,
    },  # type: ignore
)


In [83]:
def cosine_top1(mention_emb: np.ndarray, umls_emb: np.ndarray):
    mention_emb /= np.linalg.norm(mention_emb, axis=1, keepdims=True)
    umls_emb /= np.linalg.norm(umls_emb, axis=1, keepdims=True)

    sim = mention_emb @ umls_emb.T
    idx = sim.argmax(axis=1)
    score = sim[np.arange(sim.shape[0]), idx]
    return idx, score


In [84]:
import time
import psutil
import os
import torch
import tracemalloc
import gc

process = psutil.Process(os.getpid())

def cpu_rss_mb():
    return process.memory_info().rss / 1024**2

def gpu_mem_mb():
    if torch.cuda.is_available():
        return sum(
            torch.cuda.memory_allocated(i)
            for i in range(torch.cuda.device_count())
        ) / 1024**2
    return 0.0


In [85]:
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

cpu_start = cpu_rss_mb()
gpu_start = gpu_mem_mb()
start_time = time.time()

tracemalloc.start()

mention_tables = {}

for group in test_data["label"].unique():
    df_g = (
        test_data
        .filter(pl.col("label") == group)
        .select("span")
        .unique()
        .sort("span")
    )

    mentions = df_g["span"].to_list()
    embeddings = np.asarray(get_bert_embed(mentions), dtype=np.float32)

    mention_tables[group] = {
        "emb": embeddings,
        "mentions": mentions
    }
results = []

for group in mention_tables.keys():
    print(f"Processing {group}")

    m = mention_tables[group]
    u = umls_tables[group]

    idx, sim = cosine_top1(m["emb"], u["emb"])

    df_res = pl.DataFrame({
        "label": group,
        "span": m["mentions"],
        "Prediction": [u["synonyms"][i] for i in idx],
        "Predicted_code": [u["codes"][i] for i in idx],
        "Prediction_score": sim
    })

    results.append(df_res)

final_df = pl.concat(results).explode("Predicted_code")
# Stop timers
end_time = time.time()

# Memory stats
cpu_end = cpu_rss_mb()
gpu_peak = torch.cuda.max_memory_allocated() / 1024**2

current, py_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

# Speed stats
total_mentions = final_df.height
total_time = end_time - start_time
throughput = total_mentions / total_time

100%|##########| 370/370 [00:00<00:00, 7696.85it/s]


Processing Devices
Processing Phenomena
Processing Objects
Processing Physiology
Processing Anatomy
Processing Disorders
Processing Chemicals & Drugs
Processing Geographic Areas
Processing Living Beings
Processing Procedures


In [41]:
print("\n==== BENCHMARK RESULTS ====")
print(f"Total time: {total_time:.2f} s")
print(f"Throughput: {throughput:.2f} mentions/s")

print(f"CPU RAM increase: {cpu_end - cpu_start:.2f} MB")
print(f"GPU peak memory: {gpu_peak:.2f} MB")
print(f"Python heap peak: {py_peak / 1024**2:.2f} MB")



==== BENCHMARK RESULTS ====
Total time: 14.67 s
Throughput: 163.79 mentions/s
CPU RAM increase: 3.23 MB
GPU peak memory: 1233.42 MB
Python heap peak: 5690.35 MB


In [28]:
print("\n==== BENCHMARK RESULTS ====")
print(f"Total time: {total_time:.2f} s")
print(f"Throughput: {throughput:.2f} mentions/s")

print(f"CPU RAM increase: {cpu_end - cpu_start:.2f} MB")
print(f"GPU peak memory: {gpu_peak:.2f} MB")
print(f"Python heap peak: {py_peak / 1024**2:.2f} MB")



==== BENCHMARK RESULTS ====
Total time: 12.71 s
Throughput: 52.40 mentions/s
CPU RAM increase: -29.79 MB
GPU peak memory: 1170.17 MB
Python heap peak: 5463.08 MB


In [13]:
print("\n==== BENCHMARK RESULTS ====")
print(f"Total time: {total_time:.2f} s")
print(f"Throughput: {throughput:.2f} mentions/s")

print(f"CPU RAM increase: {cpu_end - cpu_start:.2f} MB")
print(f"GPU peak memory: {gpu_peak:.2f} MB")
print(f"Python heap peak: {py_peak / 1024**2:.2f} MB")



==== BENCHMARK RESULTS ====
Total time: 24.82 s
Throughput: 719.25 mentions/s
CPU RAM increase: 103.65 MB
GPU peak memory: 1234.29 MB
Python heap peak: 13138.38 MB
